In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# 1. Cargar el modelo entrenado con MNIST
model = load_model("modelo2_cnn.keras")
tablero = []

for i in range(81):
    # 2. Leer la celda guardada en escala de grises
    img = cv2.imread(f"celdas/celda_{i}.jpg", cv2.IMREAD_GRAYSCALE)
    
    # -------------------------------------------------------------------------
    # EXTRACCIÓN Y LIMPIEZA ADAPTATIVA DE CONTORNOS (Anti-descentrado de YOLO)
    # -------------------------------------------------------------------------
    # Binarizamos temporalmente la celda completa para buscar elementos
    _, img_init_bin = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Buscamos todos los contornos presentes en la celda
    contornos_padre, _ = cv2.findContours(img_init_bin, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    
    img_final = np.zeros((28, 28), dtype=np.uint8)
    encontrado = False
    w_box, h_box = 0, 0
    
    if contornos_padre:
        h_c, w_c = img.shape
        contornos_validos = []
        
        for c in contornos_padre:
            x, y, w_box_c, h_box_c = cv2.boundingRect(c)
            
            # Filtro: Si el contorno toca los bordes extremos del recorte y es gigante,
            # significa que es la línea de la cuadrícula azul/negra desalineada. Lo ignoramos.
            if x <= 1 or y <= 1 or (x + w_box_c) >= w_c - 1 or (y + h_box_c) >= h_c - 1:
                if cv2.contourArea(c) > (w_c * h_c * 0.3):
                    continue
            contornos_validos.append(c)
            
        if contornos_validos:
            # El contorno más grande de los válidos será nuestro número intacto
            c_numero = max(contornos_validos, key=cv2.contourArea)
            x, y, w_box, h_box = cv2.boundingRect(c_numero)
            
            # Si el tamaño mínimo es razonable, procedemos a extraerlo
            if w_box >= 3 and h_box >= 6:
                digito = img_init_bin[y:y+h_box, x:x+w_box]
                
                # Redimensionamos el número dándole el colchón negro simétrico de MNIST
                factor = min(18 / h_box, 18 / w_box)
                nuevo_w = int(w_box * factor)
                nuevo_h = int(h_box * factor)
                digito_scaled = cv2.resize(digito, (nuevo_w, nuevo_h))
                
                # Centramos el dígito escalado en el lienzo de 28x28
                start_y = (28 - nuevo_h) // 2
                start_x = (28 - nuevo_w) // 2
                img_final[start_y:start_y+nuevo_h, start_x:start_x+nuevo_w] = digito_scaled
                encontrado = True

    # Si no se encontró ningún número o quedó vacío tras el filtro, añadimos un 0 (Vacío)
    if not encontrado:
        tablero.append(0)
        continue
        
    # Control extra: Si el número de píxeles final es ridículo, es una celda vacía
    if cv2.countNonZero(img_final) < 20:
        tablero.append(0)
        continue

    # -------------------------------------------------------------------------
    # PROCESAMIENTO Y PREDICCIÓN CON LA CNN
    # -------------------------------------------------------------------------
    img_cnn = img_final.astype("float32") / 255.0
    img_cnn = img_cnn.reshape(1, 28, 28, 1)
    
    pred = model.predict(img_cnn, verbose=0)
    clase_predicha = np.argmax(pred)
    
    # -------------------------------------------------------------------------
    # FILTRO GEOMÉTRICO (Doble verificación para desempate del 7/1)
    # -------------------------------------------------------------------------
    # Si la red insiste en que es un 1, miramos la relación de aspecto del contorno original.
    # Un '1' tipográfico real es muy estrecho. El '7' siempre conserva más anchura arriba.
    if clase_predicha == 1 and encontrado:
        relacion_aspecto = w_box / float(h_box)
        if relacion_aspecto > 0.48:
            clase_predicha = 7
            
    tablero.append(int(clase_predicha))

# 8. Reconstruir la matriz final de 9x9 limpia
sudoku_matriz = np.array(tablero).reshape(9, 9)
print("\nMatriz del Sudoku generada con éxito:")
print(sudoku_matriz)

# --- GUARDAR LA MATRIZ PARA EL MODELO 3 ---
# Opción A: Guardarlo en formato binario de Numpy (.npy) -> Es la más limpia
np.save("sudoku_detectado.npy", sudoku_matriz)

# Opción B (Opcional): Guardarlo como un archivo de texto común (.txt) por si quieres abrirlo a mano y verlo
np.savetxt("sudoku_detectado.txt", sudoku_matriz, fmt="%d")

print("¡Matriz exportada con éxito para el Modelo 3!")


Matriz del Sudoku generada con éxito:
[[0 0 2 7 0 5 4 9 3]
 [5 0 0 0 0 0 5 0 0]
 [0 0 7 0 2 0 5 8 0]
 [0 0 5 5 3 0 8 0 0]
 [0 0 3 0 0 5 0 0 2]
 [4 7 8 0 0 9 0 0 0]
 [0 5 0 0 0 0 2 0 5]
 [3 0 0 0 4 0 0 0 0]
 [0 0 0 3 5 0 7 0 0]]
¡Matriz exportada con éxito para el Modelo 3!
